In [26]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Dense, Embedding, Flatten, Dropout,
                                      Conv1D, GlobalMaxPooling1D, GRU, LSTM)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

# 데이터 불러오기

In [27]:
import pandas as pd

df = pd.read_csv('./data/all-data.csv',
                 header=None,
                 names=['label', 'sentence'],
                 encoding='latin-1')

print(df.shape)
print(df['label'].value_counts())
print(df.head())

(4846, 2)
label
neutral     2879
positive    1363
negative     604
Name: count, dtype: int64
      label                                           sentence
0   neutral  According to Gran , the company has no plans t...
1   neutral  Technopolis plans to develop in stages an area...
2  negative  The international electronic industry company ...
3  positive  With the new production plant the company woul...
4  positive  According to the company 's updated strategy f...


In [28]:
# 결측값 확인
print(df.isnull().sum())

# 문장 길이 분포 확인
df['length'] = df['sentence'].apply(len)
print(df['length'].describe())

label       0
sentence    0
dtype: int64
count    4846.000000
mean      128.132068
std        56.526180
min         9.000000
25%        84.000000
50%       119.000000
75%       163.000000
max       315.000000
Name: length, dtype: float64


In [29]:
from sklearn.model_selection import train_test_split

X = df['sentence']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(y_train.value_counts())


Train: 3876, Test: 970
label
neutral     2303
positive    1090
negative     483
Name: count, dtype: int64


# 토크나이징 작업

In [30]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 토크나이저 설정
MAX_WORDS = 5000   # 최대 단어 수
MAX_LEN = 100      # 문장 최대 길이

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

# 텍스트 → 숫자 시퀀스 변환
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

# 길이 통일 (패딩)
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='post', truncating='post')

print(X_train_pad.shape)
print(X_test_pad.shape)
print(X_train_pad[0])  # 첫 번째 문장 숫자 변환 확인


(3876, 100)
(970, 100)
[   2  332 2431  113  123  808  577   26   20   49   39   96  418  148
   33    6 1778 1396   28 1779   14 2432 2024   28 1779    4  296    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0]


# 라벨 negative, neutral, positive 원핫인코딩

In [31]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

le = LabelEncoder()
y_train_enc = to_categorical(le.fit_transform(y_train), num_classes=3)
y_test_enc  = to_categorical(le.transform(y_test), num_classes=3)

print(le.classes_)       # 라벨 순서 확인
print(y_train_enc[0], y_train_enc[1], y_train_enc[2])    # 첫 번째 라벨 확인


['negative' 'neutral' 'positive']
[0. 0. 1.] [0. 1. 0.] [0. 0. 1.]


# MLP 모델

In [32]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Flatten, Dropout

VOCAB_SIZE = 5000
EMBED_DIM = 32

mlp_model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),  # 단어 → 벡터
    Flatten(),                                                 # 벡터 → 1차원
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')                            # 3개 클래스 출력
])

mlp_model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])


/home/ubuntu/miniforge3/envs/dl_env/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [33]:
history_mlp = mlp_model.fit(
    X_train_pad, y_train_enc,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)
mlp_model.summary()

Epoch 1/10


I0000 00:00:1775092034.549821    1761 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_51451__.20


91/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5995 - loss: 0.9544

I0000 00:00:1775092035.724919    1766 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_51451__.20


97/97 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.5932 - loss: 0.9301 - val_accuracy: 0.6302 - val_loss: 0.8513
Epoch 2/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6803 - loss: 0.7632 - val_accuracy: 0.6985 - val_loss: 0.7520
Epoch 3/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8016 - loss: 0.5127 - val_accuracy: 0.6701 - val_loss: 0.7488
Epoch 4/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8768 - loss: 0.2964 - val_accuracy: 0.7075 - val_loss: 0.8553
Epoch 5/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9690 - loss: 0.1163 - val_accuracy: 0.7384 - val_loss: 0.9227
Epoch 6/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9935 - loss: 0.0290 - val_accuracy: 0.7320 - val_loss: 1.0170
Epoch 7/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9974 - loss: 0.0121 - val_accuracy: 0.7332 - val_loss: 1.3251
Epoch 8/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9981 - loss: 0.0108 - val_accuracy: 0.7410 - val_loss: 1.2482
Ep

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_13 (Embedding)        │ (None, 100, 32)        │       160,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 3200)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 64)             │       204,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,101,131 (4.20 MB)

 Trainable params: 367,043 (1.40 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 734,088 (2.80 MB)

In [34]:
# 테스트셋 최종 평가
mlp_loss, mlp_acc = mlp_model.evaluate(X_test_pad, y_test_enc, verbose=0)

# 결과 저장용 딕셔너리
results = {}
results['MLP'] = {
    'test_accuracy': mlp_acc,
    'test_loss': mlp_loss,
    'history': history_mlp
}

print(f"MLP 테스트 정확도: {mlp_acc:.4f}")
print(f"MLP 테스트 손실:   {mlp_loss:.4f}")

MLP 테스트 정확도: 0.7268
MLP 테스트 손실:   1.3074


- MLP 원본          → val_accuracy: 72.7% (과적합)
- MLP + EarlyStopping→ ?

In [35]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',      # 검증 손실 기준
    patience=3,              # 3번 연속 안 좋아지면 종료
    restore_best_weights=True # 가장 좋았던 시점으로 복원
)

mlp_model2 = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

mlp_model2.compile(optimizer='adam',
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])

history_mlp2 = mlp_model2.fit(
    X_train_pad, y_train_enc,
    epochs=50,              # 넉넉히 설정, EarlyStopping이 알아서 멈춤
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/50


I0000 00:00:1775092041.141544    1768 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_58871__.20


82/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5544 - loss: 0.9857

I0000 00:00:1775092042.272171    1761 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_58871__.20


97/97 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.5839 - loss: 0.9397 - val_accuracy: 0.6070 - val_loss: 0.8739
Epoch 2/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6429 - loss: 0.8050 - val_accuracy: 0.6778 - val_loss: 0.7450
Epoch 3/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7752 - loss: 0.5535 - val_accuracy: 0.6830 - val_loss: 0.7072
Epoch 4/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9181 - loss: 0.2415 - val_accuracy: 0.7281 - val_loss: 0.7980
Epoch 5/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9810 - loss: 0.0729 - val_accuracy: 0.7255 - val_loss: 0.9577
Epoch 6/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9942 - loss: 0.0265 - val_accuracy: 0.7345 - val_loss: 1.0180


In [36]:
mlp_loss, mlp_acc = mlp_model2.evaluate(X_test_pad, y_test_enc, verbose=0)

results = {}
results['MLP'] = {
    'test_accuracy': round(mlp_acc, 4),
    'test_loss': round(mlp_loss, 4)
}

print(f"MLP | 정확도: {mlp_acc:.4f} | 손실: {mlp_loss:.4f}")

MLP | 정확도: 0.6918 | 손실: 0.7175


- MLP는 Epoch 2에서 이미 한계에 도달함.
- 그 이후로는 훈련 데이터만 외우고(과적합) 새로운 데이터엔 오히려 성능이 떨어짐.

# CNN 모델

In [37]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Embedding, Flatten, Dropout, Conv1D, GlobalMaxPooling1D

early_stop_cnn = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

cnn_model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
    Conv1D(64, kernel_size=3, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

cnn_model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

history_cnn = cnn_model.fit(
    X_train_pad, y_train_enc,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop_cnn],
    verbose=1
)


Epoch 1/50


I0000 00:00:1775092046.717380    1766 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_64383__.22


90/97 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5367 - loss: 0.9922

I0000 00:00:1775092048.148111    1763 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_64383__.22


97/97 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.5794 - loss: 0.9441 - val_accuracy: 0.6070 - val_loss: 0.8773
Epoch 2/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6603 - loss: 0.7875 - val_accuracy: 0.7075 - val_loss: 0.7049
Epoch 3/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7868 - loss: 0.5432 - val_accuracy: 0.7294 - val_loss: 0.6422
Epoch 4/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8861 - loss: 0.3187 - val_accuracy: 0.7925 - val_loss: 0.6544
Epoch 5/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9532 - loss: 0.1629 - val_accuracy: 0.7745 - val_loss: 0.6672
Epoch 6/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9816 - loss: 0.0789 - val_accuracy: 0.7822 - val_loss: 0.7711


In [38]:
cnn_loss, cnn_acc = cnn_model.evaluate(X_test_pad, y_test_enc, verbose=0)

results['CNN'] = {
    'test_accuracy': round(cnn_acc, 4),
    'test_loss': round(cnn_loss, 4)
}

print(f"MLP | 정확도: {results['MLP']['test_accuracy']} | 손실: {results['MLP']['test_loss']}")
print(f"CNN | 정확도: {results['CNN']['test_accuracy']} | 손실: {results['CNN']['test_loss']}")



MLP | 정확도: 0.6918 | 손실: 0.7175
CNN | 정확도: 0.733 | 손실: 0.694


# GRU

In [39]:
from tensorflow.keras.layers import (Dense, Embedding, Flatten, Dropout,
                                      Conv1D, GlobalMaxPooling1D, GRU, LSTM)

early_stop_gru = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

gru_model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
    GRU(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

gru_model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

history_gru = gru_model.fit(
    X_train_pad, y_train_enc,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop_gru],
    verbose=1
)


Epoch 1/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.5865 - loss: 0.9463 - val_accuracy: 0.6070 - val_loss: 0.9202
Epoch 2/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5910 - loss: 0.9342 - val_accuracy: 0.6070 - val_loss: 0.9206
Epoch 3/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5910 - loss: 0.9328 - val_accuracy: 0.6070 - val_loss: 0.9188
Epoch 4/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - accuracy: 0.5910 - loss: 0.9340 - val_accuracy: 0.6070 - val_loss: 0.9169
Epoch 5/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.5910 - loss: 0.9309 - val_accuracy: 0.6070 - val_loss: 0.9179
Epoch 6/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5910 - loss: 0.9313 - val_accuracy: 0.6070 - val_loss: 0.9187
Epoch 7/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5910 - loss: 0.9307 - val_accuracy: 0.6070 - val_loss: 0.9178


In [40]:
gru_loss, gru_acc = gru_model.evaluate(X_test_pad, y_test_enc, verbose=0)

results['GRU'] = {
    'test_accuracy': round(gru_acc, 4),
    'test_loss': round(gru_loss, 4)
}

print(f"MLP | 정확도: {results['MLP']['test_accuracy']} | 손실: {results['MLP']['test_loss']}")
print(f"CNN | 정확도: {results['CNN']['test_accuracy']} | 손실: {results['CNN']['test_loss']}")
print(f"GRU | 정확도: {results['GRU']['test_accuracy']} | 손실: {results['GRU']['test_loss']}")


MLP | 정확도: 0.6918 | 손실: 0.7175
CNN | 정확도: 0.733 | 손실: 0.694
GRU | 정확도: 0.5938 | 손실: 0.9269


# LSTM 

In [41]:
from tensorflow.keras.layers import LSTM

early_stop_lstm = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

lstm_model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

lstm_model.compile(optimizer='adam',
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])

history_lstm = lstm_model.fit(
    X_train_pad, y_train_enc,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop_lstm],
    verbose=1
)


Epoch 1/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.5781 - loss: 0.9634 - val_accuracy: 0.6070 - val_loss: 0.9177
Epoch 2/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5910 - loss: 0.9338 - val_accuracy: 0.6070 - val_loss: 0.9171
Epoch 3/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.5910 - loss: 0.9342 - val_accuracy: 0.6070 - val_loss: 0.9192
Epoch 4/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5910 - loss: 0.9325 - val_accuracy: 0.6070 - val_loss: 0.9169
Epoch 5/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.5910 - loss: 0.9349 - val_accuracy: 0.6070 - val_loss: 0.9178
Epoch 6/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.5910 - loss: 0.9329 - val_accuracy: 0.6070 - val_loss: 0.9253
Epoch 7/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5910 - loss: 0.9320 - val_accuracy: 0.6070 - val_loss: 0.9210


RNN 64로 올려서 학습 다시

In [42]:
EMBED_DIM = 64  # 32 → 64

early_stop_lstm = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

lstm_model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

lstm_model.compile(optimizer='adam',
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])

history_lstm = lstm_model.fit(
    X_train_pad, y_train_enc,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop_lstm],
    verbose=1
)


Epoch 1/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.5852 - loss: 0.9467 - val_accuracy: 0.6070 - val_loss: 0.9313
Epoch 2/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5910 - loss: 0.9339 - val_accuracy: 0.6070 - val_loss: 0.9219
Epoch 3/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5910 - loss: 0.9345 - val_accuracy: 0.6070 - val_loss: 0.9209
Epoch 4/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5910 - loss: 0.9345 - val_accuracy: 0.6070 - val_loss: 0.9234
Epoch 5/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5910 - loss: 0.9319 - val_accuracy: 0.6070 - val_loss: 0.9255
Epoch 6/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.5910 - loss: 0.9310 - val_accuracy: 0.6070 - val_loss: 0.9183
Epoch 7/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5910 - loss: 0.9323 - val_accuracy: 0.6070 - val_loss: 0.9186
Epoch 8/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5910 - loss: 0.9311 - val_accuracy: 0.6070 - v

Input_length 제거

In [43]:
early_stop_lstm = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

lstm_model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM),   # input_length 제거
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

lstm_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),  # lr 높임
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])

history_lstm = lstm_model.fit(
    X_train_pad, y_train_enc,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop_lstm],
    verbose=1
)


Epoch 1/50


97/97 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.5832 - loss: 0.9542 - val_accuracy: 0.6070 - val_loss: 0.9180
Epoch 2/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5910 - loss: 0.9360 - val_accuracy: 0.6070 - val_loss: 0.9222
Epoch 3/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5910 - loss: 0.9337 - val_accuracy: 0.6070 - val_loss: 0.9219
Epoch 4/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5910 - loss: 0.9322 - val_accuracy: 0.6070 - val_loss: 0.9177
Epoch 5/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5910 - loss: 0.9304 - val_accuracy: 0.6070 - val_loss: 0.9199
Epoch 6/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5910 - loss: 0.9299 - val_accuracy: 0.6070 - val_loss: 0.9203
Epoch 7/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5910 - loss: 0.9303 - val_accuracy: 0.6070 - val_loss: 0.9196
Epoch 8/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5910 - loss: 0.9306 - val_accuracy: 0.6070 - val_loss: 0.

# MLP, CNN, GRU, LSTM 비교

In [44]:
lstm_loss, lstm_acc = lstm_model.evaluate(X_test_pad, y_test_enc, verbose=0)

results['LSTM'] = {
    'test_accuracy': round(lstm_acc, 4),
    'test_loss': round(lstm_loss, 4),
    'note': '학습 실패 - 환경 이슈 추정'
}

print("===== 최종 결과 비교 =====")
print(f"MLP  | 정확도: {results['MLP']['test_accuracy']} | 손실: {results['MLP']['test_loss']}")
print(f"CNN  | 정확도: {results['CNN']['test_accuracy']} | 손실: {results['CNN']['test_loss']}")
print(f"GRU  | 정확도: {results['GRU']['test_accuracy']} | 손실: {results['GRU']['test_loss']} | 학습 실패")
print(f"LSTM | 정확도: {results['LSTM']['test_accuracy']} | 손실: {results['LSTM']['test_loss']} | 학습 실패")
print(f"\n최고 성능: CNN ({results['CNN']['test_accuracy']})")


===== 최종 결과 비교 =====
MLP  | 정확도: 0.6918 | 손실: 0.7175
CNN  | 정확도: 0.733 | 손실: 0.694
GRU  | 정확도: 0.5938 | 손실: 0.9269 | 학습 실패
LSTM | 정확도: 0.5938 | 손실: 0.9261 | 학습 실패

최고 성능: CNN (0.733)


# 금융뉴스로 성능 테스트

In [45]:
from newspaper import Article

urls = [
    "https://finance.yahoo.com/sectors/energy/articles/no-other-choice-war-hit-031518842.html",
    "https://finance.yahoo.com/news/live/stock-market-today-dow-soars-1000-points-sp-500-and-nasdaq-surge-in-upbeat-end-to-brutal-quarter-184154822.html",
    "https://finance.yahoo.com/news/oil-plunges-stocks-soar-after-iranian-president-offers-first-signs-of-willingness-to-end-war-with-us-183158792.html",
    "https://finance.yahoo.com/news/airlines-face-price-hikes-lower-margins-as-iran-war-pressures-business-171745222.html"
]

full_texts = []
titles = []
for url in urls:
    try:
        article = Article(url)
        article.download()
        article.parse()
        full_texts.append(article.text)
        titles.append(article.title)
        print(f"완료: {article.title}")
    except Exception as e:
        print(f"실패: {url}")


완료: ‘There’s No Other Choice’: War-Hit Asian Buyers Grab Russian Oil
완료: Stock market today: Dow soars 1,000 points, S&P 500 and Nasdaq surge in upbeat end to brutal quarter
완료: Oil plunges, stocks soar after Iranian president offers first signs of willingness to end war with US
완료: Airlines face price hikes, lower margins as Iran war pressures business


## 본문 전체로 예측

In [47]:
def predict_sentiment(sentences):
    seq = tokenizer.texts_to_sequences(sentences)
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
    pred = cnn_model.predict(pad, verbose=0)
    labels = le.classes_
    
    for s, p in zip(sentences, pred):
        result = labels[np.argmax(p)]
        print(f"문장: {s[:50]}...")
        print(f"예측: {result} (negative: {p[0]:.2f}, neutral: {p[1]:.2f}, positive: {p[2]:.2f})")
        print()

In [52]:
def get_prediction(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
    pred = cnn_model.predict(pad, verbose=0)[0]
    labels = le.classes_
    return labels[np.argmax(pred)], pred


In [48]:
from newspaper import Article
import numpy as np

urls = [
    "https://finance.yahoo.com/sectors/energy/articles/no-other-choice-war-hit-031518842.html",
    "https://finance.yahoo.com/news/live/stock-market-today-dow-soars-1000-points-sp-500-and-nasdaq-surge-in-upbeat-end-to-brutal-quarter-184154822.html",
    "https://finance.yahoo.com/news/oil-plunges-stocks-soar-after-iranian-president-offers-first-signs-of-willingness-to-end-war-with-us-183158792.html",
    "https://finance.yahoo.com/news/airlines-face-price-hikes-lower-margins-as-iran-war-pressures-business-171745222.html"
]

full_texts = []
for url in urls:
    try:
        article = Article(url)
        article.download()
        article.parse()
        full_texts.append(article.text)
    except:
        print(f"실패: {url}")

predict_sentiment(full_texts)


문장: (Bloomberg) — Energy-starved Asian nations are tak...
예측: neutral (negative: 0.08, neutral: 0.64, positive: 0.28)

문장: US stocks surged on Tuesday after the Iranian pres...
예측: positive (negative: 0.10, neutral: 0.06, positive: 0.84)

문장: Oil prices plunged on Tuesday afternoon on signals...
예측: neutral (negative: 0.14, neutral: 0.55, positive: 0.31)

문장: Airlines are scrambling to protect their business ...
예측: positive (negative: 0.13, neutral: 0.41, positive: 0.47)



In [49]:
def predict_sentiment_pretty(texts, titles=None):
    for i, text in enumerate(texts):
        seq = tokenizer.texts_to_sequences([text])
        pad = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
        pred = cnn_model.predict(pad, verbose=0)[0]
        labels = le.classes_

        result = labels[np.argmax(pred)]
        emoji = {"positive": "🟢", "neutral": "🟡", "negative": "🔴"}

        print(f"{'='*60}")
        if titles:
            print(f"제목: {titles[i]}")
        print(f"예측: {emoji[result]} {result.upper()}")
        print(f"  negative: {pred[0]:.2%}")
        print(f"  neutral:  {pred[1]:.2%}")
        print(f"  positive: {pred[2]:.2%}")
        print()

predict_sentiment_pretty(full_texts, titles=titles)

제목: ‘There’s No Other Choice’: War-Hit Asian Buyers Grab Russian Oil
예측: 🟡 NEUTRAL
  negative: 7.94%
  neutral:  63.76%
  positive: 28.30%

제목: Stock market today: Dow soars 1,000 points, S&P 500 and Nasdaq surge in upbeat end to brutal quarter
예측: 🟢 POSITIVE
  negative: 10.11%
  neutral:  6.24%
  positive: 83.65%

제목: Oil plunges, stocks soar after Iranian president offers first signs of willingness to end war with US
예측: 🟡 NEUTRAL
  negative: 14.03%
  neutral:  54.59%
  positive: 31.38%

제목: Airlines face price hikes, lower margins as Iran war pressures business
예측: 🟢 POSITIVE
  negative: 12.62%
  neutral:  40.63%
  positive: 46.74%



## 헤드라인 + 본문 첫 2~3문장

In [50]:
from nltk.tokenize import sent_tokenize

summary_texts = []
for text in full_texts:
    first_3 = " ".join(sent_tokenize(text)[:3])
    summary_texts.append(first_3)

predict_sentiment_pretty(summary_texts, titles=titles)
from nltk.tokenize import sent_tokenize

summary_texts = []
for text in full_texts:
    first_3 = " ".join(sent_tokenize(text)[:3])
    summary_texts.append(first_3)

predict_sentiment_pretty(summary_texts, titles=titles)

제목: ‘There’s No Other Choice’: War-Hit Asian Buyers Grab Russian Oil
예측: 🟡 NEUTRAL
  negative: 6.86%
  neutral:  70.47%
  positive: 22.67%

제목: Stock market today: Dow soars 1,000 points, S&P 500 and Nasdaq surge in upbeat end to brutal quarter
예측: 🟢 POSITIVE
  negative: 10.11%
  neutral:  6.24%
  positive: 83.65%

제목: Oil plunges, stocks soar after Iranian president offers first signs of willingness to end war with US
예측: 🟡 NEUTRAL
  negative: 14.03%
  neutral:  54.59%
  positive: 31.38%

제목: Airlines face price hikes, lower margins as Iran war pressures business
예측: 🟡 NEUTRAL
  negative: 7.88%
  neutral:  73.23%
  positive: 18.89%

제목: ‘There’s No Other Choice’: War-Hit Asian Buyers Grab Russian Oil
예측: 🟡 NEUTRAL
  negative: 6.86%
  neutral:  70.47%
  positive: 22.67%

제목: Stock market today: Dow soars 1,000 points, S&P 500 and Nasdaq surge in upbeat end to brutal quarter
예측: 🟢 POSITIVE
  negative: 10.11%
  neutral:  6.24%
  positive: 83.65%

제목: Oil plunges, stocks soar after Irania

In [53]:
print(f"{'제목':<45} {'본문전체':^25} {'앞3문장':^25}")
print("="*95)
for i in range(len(titles)):
    title = titles[i][:40]
    result_a, pred_a = get_prediction(full_texts[i])
    result_c, pred_c = get_prediction(summary_texts[i])
    
    a_str = f"{result_a}({pred_a[np.argmax(pred_a)]:.0%})"
    c_str = f"{result_c}({pred_c[np.argmax(pred_c)]:.0%})"
    
    print(f"{title:<45} {a_str:^25} {c_str:^25}")


제목                                                      본문전체                      앞3문장           
‘There’s No Other Choice’: War-Hit Asian            neutral(64%)              neutral(70%)       
Stock market today: Dow soars 1,000 poin            positive(84%)             positive(84%)      
Oil plunges, stocks soar after Iranian p            neutral(55%)              neutral(55%)       
Airlines face price hikes, lower margins            positive(47%)             neutral(73%)       
